### **02613 Mini-Project: Wall Heating**

### **``run_subset.py``**

In [ ]:
def load_building_ids(load_dir):
    with open(join(load_dir, "building_ids.txt"), "r") as f:
        return f.read().splitlines()

The loading of the ``building_ids`` takes a `txt`-file as input and splits the string such that each line is a list item. 

In [ ]:
def load_data(load_dir, bid):
    """
    Loads one building:
      - domain: 512x512 initial temperature grid
      - interior mask: 512x512 boolean mask
    Pads the domain to 514x514 to match the reference implementation.
    """
    u = np.zeros((GRID_SIZE + 2, GRID_SIZE + 2), dtype=np.float64)
    u[1:-1, 1:-1] = np.load(join(load_dir, f"{bid}_domain.npy"))
    interior_mask = np.load(join(load_dir, f"{bid}_interior.npy")).astype(bool)
    return u, interior_mask

The loading of a single building floor plan. It has the $512 \times 512$ temperature grid with a boundary layer on each side (padding), making it a $514 \times 514$ grid.

In [ ]:
def jacobi(u, interior_mask, max_iter=20_000, atol=1e-4):
    """
    Reference-style Jacobi solver.
    """
    u = np.copy(u)

    for _ in range(max_iter):
        u_new = 0.25 * (
            u[1:-1, :-2] +   # left
            u[1:-1, 2:] +    # right
            u[:-2, 1:-1] +   # up
            u[2:, 1:-1]      # down
        )

        u_old_interior = u[1:-1, 1:-1][interior_mask]
        u_new_interior = u_new[interior_mask]

        delta = np.abs(u_old_interior - u_new_interior).max()
        u[1:-1, 1:-1][interior_mask] = u_new_interior

        if delta < atol:
            break

    return u

The Jacobi method creates a copy of the original array using ``np.copy(u)`` in order to keep an unmodified version. Each iteration computes the mean of an element's neighbours (up, down, left, and right). ``interior_mask`` selects which interior points are allowed to be altered; while ``u_new`` is computed for the entire array, but only the masked elements are updated.

The ``u_old_interior`` extracts the current values at the active (masked) interior points, and the ``u_new_interior`` extracts the newly computed Jacobi values at those same points. ``u[1:-1, ]`` takes rows from 1 to the second last one (excluding 0 and last, i.e., removing boundaries).

``delta`` finds the largest change in any updated point during the current iteration. This is the convergence check, i.e., if the largest (temperature) change becomes small enough, then the sequence is said to have converged.

The next line writes the newly computed values at masked points back into ``u`` to be used for the next iteration.

The convergence check is ``delta < atol``, where we break if ``delta`` is smaller than a predefined tolerance.

In [ ]:
def summary_stats(u, interior_mask):
    u_interior = u[1:-1, 1:-1][interior_mask]

    mean_temp = u_interior.mean()
    std_temp = u_interior.std()
    pct_above_18 = np.mean(u_interior > 18.0) * 100.0
    pct_below_15 = np.mean(u_interior < 15.0) * 100.0

    return {
        "mean_temp": mean_temp,
        "std_temp": std_temp,
        "pct_above_18": pct_above_18,
        "pct_below_15": pct_below_15,
    }

The ``summary_stats()`` function returns the mean and standard deviation of the temperature in the building, as well as the percentage of area above 18 degrees and below 15 degrees, respectively.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Run wall-heating simulation on first N floorplans.")
    parser.add_argument("N", type=int, nargs="?", default=1, help="Number of floorplans to process")
    parser.add_argument("--max-iter", type=int, default=20_000, help="Maximum Jacobi iterations")
    parser.add_argument("--atol", type=float, default=1e-4, help="Convergence tolerance")
    parser.add_argument("--time", action="store_true", help="Print timing information")
    args = parser.parse_args()

    building_ids = load_building_ids(LOAD_DIR)[:args.N]

    if args.time:
        t0 = time.perf_counter()

    print("building_id,mean_temp,std_temp,pct_above_18,pct_below_15")

    for bid in building_ids:
        u0, interior_mask = load_data(LOAD_DIR, bid)
        u = jacobi(u0, interior_mask, max_iter=args.max_iter, atol=args.atol)
        stats = summary_stats(u, interior_mask)

        print(
            f"{bid},"
            f"{stats['mean_temp']},"
            f"{stats['std_temp']},"
            f"{stats['pct_above_18']},"
            f"{stats['pct_below_15']}"
        )

    if args.time:
        t1 = time.perf_counter()
        print(f"# Total runtime: {t1 - t0:.3f} seconds")


if __name__ == "__main__":
    main()

The parser arguments are command-line arguments: values you can pass when running the Python script from the terminal. In bash: 

``python script.py 4 --max-iter 10000 --atol 1e-5 --time`` 

takes 4 floorplans, runs maximum 10000 Jacobi iterations with a convergence tolerance of 0.00001 and prints timing information.

#### **How to run the code**

Log in to the hpc: ``ssh s225141@login.hpc.dtu.dk``

Access the project working directory created: ``cd ~/wall_heating``

Run the script: ``python script.py 2 --max-iter 10000 --atol 1e-5 --time``

___

Profiling the ``jacobi`` function using kernprof means adding ``@profile`` above the function definition.

If profiling stops working, then run in bash: ``source ~/venv_hpc/bin/activate
pip install line_profiler``

Then run: ``kernprof -l run_subset.py 10``

The program writes profile results to ``run_subset.py.lprof``

Inspect results with:

``python -m line_profiler -rmt run_subset.py.lprof``



___

#### **``visualise_results.py``**

Adds the following to the script ``run_subset_2.py``:

In [ ]:
def make_plot(bid, u0, interior_mask, u_final, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    im0 = axes[0].imshow(u0[1:-1, 1:-1], origin="lower")
    axes[0].set_title(f"Initial grid\nBuilding {bid}")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(interior_mask, origin="lower")
    axes[1].set_title("Interior mask")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(u_final[1:-1, 1:-1], origin="lower")
    axes[2].set_title("Final temperature")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    for ax in axes:
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.tight_layout()
    out_path = join(out_dir, f"{bid}_visualization.png")
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_path}")

Log in to the hpc: ``ssh s225141@login.hpc.dtu.dk``

Access the project working directory created: ``cd ~/wall_heating``

If ``matplotlib`` stops working, then run the following in bash inside the hpc: 

``source ~/venv_hpc/bin/activate
pip install matplotlib``

Then run in bash: ``python visualise_results.py 3``

That will create a folder called figures with files like: 

``figures/10000_visualization.png

figures/10009_visualization.png

figures/10014_visualization.png``

To see them on the cluster, use: ``ls figures``

Then copy them back to your laptop with scp, for example: ``scp s225141@login.hpc.dtu.dk:~/wall_heating/figures/*.png .``

___

### **``run_subset_5.py``**

In [ ]:
def process_building(bid, max_iter, atol):
    u0, interior_mask = load_data(LOAD_DIR, bid)
    u = jacobi(u0, interior_mask, max_iter=max_iter, atol=atol)
    stats = summary_stats(u, interior_mask)

    return (
        bid,
        stats["mean_temp"],
        stats["std_temp"],
        stats["pct_above_18"],
        stats["pct_below_15"],
    )

This wraps the whole workflow for one building:

- load data
- run Jacobi
- compute stats
- return one result row

That makes it easy for each worker process to handle buildings independently.

___

In [ ]:
def chunk_list(items, n_chunks):
    n = len(items)
    chunk_size = math.ceil(n / n_chunks)
    return [items[i:i + chunk_size] for i in range(0, n, chunk_size)]

This is the core of static scheduling. It splits the selected building IDs into fixed chunks, so each worker gets roughly the same number of floorplans. Example:

- 50 buildings
- 4 workers

might become chunks of size:

- 13, 13, 12, 12

Each worker gets one chunk and keeps it for the whole run.

___

In [ ]:
def worker_process_chunk(args):
    chunk, max_iter, atol = args
    results = []

    for bid in chunk:
        results.append(process_building(bid, max_iter, atol))

    return results

This is what each worker process actually runs. Each worker receives:

- one fixed chunk of building IDs
- ``max_iter``
- ``atol``

Then it processes its assigned buildings sequentially and returns the results. This is exactly what makes it static scheduling instead of dynamic scheduling.

___

The main loop has been changed from the original:

In [ ]:
print("building_id,mean_temp,std_temp,pct_above_18,pct_below_15")

for bid in building_ids:
    u0, interior_mask = load_data(LOAD_DIR, bid)
    u = jacobi(u0, interior_mask, max_iter=args.max_iter, atol=args.atol)
    stats = summary_stats(u, interior_mask)

    print(
        f"{bid},"
        f"{stats['mean_temp']},"
        f"{stats['std_temp']},"
        f"{stats['pct_above_18']},"
        f"{stats['pct_below_15']}"
    )

To the following:

In [ ]:
n_workers = min(args.workers, len(building_ids))
chunks = chunk_list(building_ids, n_workers)

if n_workers == 1:
    chunk_results = [worker_process_chunk((chunks[0], args.max_iter, args.atol))]
else:
    with mp.Pool(processes=n_workers) as pool:
        chunk_results = pool.map(
            worker_process_chunk,
            [(chunk, args.max_iter, args.atol) for chunk in chunks]
        )

all_results = [row for chunk in chunk_results for row in chunk]

print("building_id,mean_temp,std_temp,pct_above_18,pct_below_15")
for bid, mean_temp, std_temp, pct_above_18, pct_below_15 in all_results:
    print(f"{bid},{mean_temp},{std_temp},{pct_above_18},{pct_below_15}")

This is the main change. Instead of processing buildings one by one in the main process, we now:

1. choose the number of workers
2. split building IDs into chunks
3. give each chunk to one worker process
4. collect all returned results
5. print them afterward

So the computation is parallelized over floorplans.

___

New command-line argument ``--workers`` controls how many worker processes are launched for the speedup experiments.

In [ ]:
parser.add_argument("--workers", type=int, default=1,
                    help="Number of worker processes")

___

### **Summary**

For Task 5, we parallelized across floorplans rather than within a single floorplan, since each building simulation is independent. Static scheduling was implemented by partitioning the selected building IDs into fixed contiguous chunks and assigning one chunk to each worker process. This ensures that each worker receives approximately the same number of floorplans.

The important logical changes are:

- added multiprocessing support
- added a way to split buildings into fixed chunks
- added a worker function that handles one chunk
- replaced the sequential building loop with a process pool
- added ``--workers`` to control parallelism

The original sequential loop over floorplans was replaced by a multiprocessing-based implementation where the selected building IDs are divided into fixed chunks and distributed among worker processes. Each worker processes its assigned chunk sequentially using the original NumPy-based Jacobi solver. This implements static scheduling, since the allocation of floorplans to workers is decided before execution and does not change during runtime.

___
#### **How to run the code**

Log in to the hpc: ``ssh s225141@login.hpc.dtu.dk``

Log in to a compute node: ``linuxsh``

Access the project working directory created: ``cd ~/wall_heating``

Then run similar to previous version: ``python run_subset_5.py 10 --workers 1 --time``

___

### **``run_subset_6.py``**

**What changed from Task 5?**
- Removed static chunking

You no longer need:

``chunk_list(...)``

``worker_process_chunk(...)``

because workers are no longer assigned fixed chunks in advance.

Kept ``process_building(...)``. This still makes sense, because one task = one building.

___

The static scheduling version of the pool call changed from:

In [ ]:
chunk_results = pool.map(
    worker_process_chunk,
    [(chunk, args.max_iter, args.atol) for chunk in chunks]
)

To a dynamic scheduling version:

In [ ]:
all_results = pool.starmap(
    process_building,
    [(bid, args.max_iter, args.atol) for bid in building_ids],
    chunksize=1
)

Now each worker gets one building at a time. When a worker finishes, it immediately gets the next building from the queue.

That is exactly the dynamic behavior Task 6 is asking for.

___

In the static scheduling version, each worker was assigned a fixed subset of floorplans before execution began. Since different floorplans require different numbers of Jacobi iterations to converge, this can lead to load imbalance, where some workers finish early and remain idle while others continue computing. In the dynamic scheduling version, floorplans are assigned to workers one at a time using ``multiprocessing.Pool.starmap(..., chunksize=1)``, so workers receive new tasks as soon as they finish their current one. This improves load balancing and is expected to reduce runtime when convergence times vary between floorplans.

___

#### **How to run the code**

Log in to the hpc: ``ssh s225141@login.hpc.dtu.dk``

Access a compute node: ``linuxsh``

Access the project working directory created: ``cd ~/wall_heating``

Then run using the same number of buildings and workers as in Task 5, so the comparison is fair:

___

### **``run_subset_7.py``**

In [ ]:
@njit(cache=True)
def jacobi_numba(u0, interior_mask, max_iter, atol):
    """
    Jacobi solver using explicit loops and double buffering.
    u0:           shape (514, 514)
    interior_mask shape (512, 512)
    """
    u = u0.copy()
    u_new = u0.copy()

    nrows, ncols = u.shape

    for _ in range(max_iter):
        delta = 0.0

        # Loop over padded grid interior: 1..512 in both directions
        for i in range(1, nrows - 1):
            ii = i - 1  # corresponding row in 512x512 mask

            # j as inner loop is cache-friendly for row-major arrays
            for j in range(1, ncols - 1):
                jj = j - 1

                if interior_mask[ii, jj]:
                    new_val = 0.25 * (
                        u[i, j - 1] +
                        u[i, j + 1] +
                        u[i - 1, j] +
                        u[i + 1, j]
                    )

                    diff = abs(new_val - u[i, j])
                    if diff > delta:
                        delta = diff

                    u_new[i, j] = new_val

        u, u_new = u_new, u

        if delta < atol:
            break

    return u

Changes: 

``@njit(cache=True)`` ensures that Numba compiles it to fast machine code. (C-like speed)

The previous version would essentially create a new 512x512 array for each neighbour, sum them, multiply 0.25 to create another one. That's several large 512x512 arrays per iteration. This problem is **memory-bound**, not compute-bound.

So, moving data in memory is expensive, as well as allocating arrays. Cache gets stressed when we spend more time moving data than computing. Since this Numba double loop runs in compiled machine code (C-like speed), it creates **no temporary arrays**, **computes one value at a time**, and has **better cache usage**.

#### **How to run the code**

Log in to the hpc: ``ssh s225141@login.hpc.dtu.dk``

Access the project working directory created: ``cd ~/wall_heating``

If ``numba`` stops working, then run the following in bash inside the hpc: 

``python3 -m venv ~/venv_hpc
source ~/venv_hpc/bin/activate
pip install numba numpy``

Then run similar to previous version: ``python run_subset_2.py 10 --time``